# 01 - Data Understanding & Data Audit (Phase 1)

Production-style dataset audit for:
- `data/train_test.csv` (training data)
- `validation.csv` (unseen validation data)

**No preprocessing, feature engineering, model training, or prediction is performed.**


In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

from src.data_loader import load_datasets
from src.data_profiler import build_profile
from src.data_validator import validate_all, ValidationIssue

PROJECT_ROOT = Path('.').resolve()
REPORTS_DIR = PROJECT_ROOT / 'reports'
FIGURES_DIR = PROJECT_ROOT / 'figures'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
datasets = load_datasets(train_path='data/train_test.csv', validation_path='data/validation.csv')
train_df = datasets.train
validation_df = datasets.validation

print('data/train_test.csv shape:', train_df.shape)
print('validation.csv shape:', validation_df.shape)


In [ ]:
train_profile = build_profile(train_df)
val_profile = build_profile(validation_df)

audit = validate_all(train_df=train_df, validation_df=validation_df, load_id_col='load_id')
train_schema = audit['train_schema']
val_schema = audit['validation_schema']

train_issues = audit['train_issues']
val_issues = audit['validation_issues']
cross_schema_issues = audit['cross_dataset_schema_issues']

print('Inferred target (heuristic):', train_profile.inferred_target)
print('Inferred identifiers (heuristic):', train_profile.inferred_identifier_columns)


In [ ]:
def issues_to_markdown(issues: list[ValidationIssue]) -> str:
    if not issues:
        return '- None detected.'
    lines: list[str] = []
    for issue in issues:
        col = issue.column if issue.column is not None else '(dataset)'
        lines.append('- **[' + issue.severity + ']** ' + issue.issue_type + ' | column=' + col + ': ' + issue.message)
    return '\n'.join(lines)

def schema_md(schema_obj) -> str:
    cols = ', '.join(schema_obj.columns)
    dtypes_lines = [c + ': ' + schema_obj.dtypes[c] for c in schema_obj.columns]
    return (
        'Rows: ' + str(len(schema_obj.columns)) + '\n' +
        'Columns: ' + str(len(schema_obj.columns)) + '\n' +
        'Memory usage (deep): ' + str(schema_obj.memory_bytes) + ' bytes\n' +
        'Columns list: ' + cols + '\n' +
        'Dtypes:\n' + '\n'.join(dtypes_lines)
    )


In [ ]:
# Build Feature Dictionary (Phase 1 required deliverable)
def build_feature_dictionary_df(profile, dataset_label: str) -> pd.DataFrame:
    rows = []
    for audit in profile.feature_audits:
        if audit.possible_purpose == 'target':
            recommended = 'Model target'
        elif audit.possible_purpose == 'identifier':
            recommended = 'Use as join key only; usually not a model input'
        elif audit.possible_purpose == 'datetime':
            recommended = 'Use for time features; keep raw until Phase 2'
        elif audit.possible_purpose == 'numeric':
            recommended = 'Numeric feature (validate ranges; scale/transform in Phase 2)'
        elif audit.possible_purpose == 'categorical':
            recommended = 'Categorical feature (encode/clean in Phase 2)'
        else:
            recommended = 'Unknown'

        desc = audit.possible_purpose + ' feature inferred from name/type.'
        rows.append({
            'Feature Name': audit.feature_name,
            'Type': audit.dtype,
            'Description (inferred)': desc,
            'Missing %': f'{audit.missing_pct:.3f}%',
            'Recommended Usage': recommended,
            'Dataset': dataset_label,
        })

    return pd.DataFrame(rows)

train_dict_df = build_feature_dictionary_df(train_profile, 'train')

# Use training features as the main dictionary deliverable.
feature_dictionary_md = (
    '# Feature Dictionary (inferred, Phase 1)\n\n'
    + 'Generated from data/train_test.csv with heuristic descriptions based on inferred datatypes and column names.\n\n'
    + train_dict_df.drop(columns=['Dataset']).to_markdown(index=False)
)


In [ ]:
# Build per-feature inspection markdown (Phase 1 requirement)
feature_inspection_lines: list[str] = []
for fa in train_profile.feature_audits:
    obs = '; '.join(fa.observations) if fa.observations else 'none'
    feature_inspection_lines.append(
        '- **' + fa.feature_name + '** | dtype=' + fa.dtype + ' | missing=' + str(round(fa.missing_pct, 3)) + '% | purpose=' + fa.possible_purpose + ' | observations=' + obs
    )

feature_inspection_md = '## Inspect every feature individually (train)\n' + '\n'.join(feature_inspection_lines)


In [ ]:
# Statistical summary markdown (Phase 1 requirement)
if train_profile.numeric_summary.empty:
    numeric_summary_md = '- No numeric columns detected.'
else:
    numeric_summary_md = train_profile.numeric_summary.to_markdown()

if train_profile.categorical_summary.empty:
    categorical_summary_md = '- No categorical/object/string columns detected.'
else:
    categorical_summary_md = train_profile.categorical_summary.reset_index().to_markdown(index=False)

numeric_md = '## Numerical statistical summary (train)\n' + numeric_summary_md
categorical_md = '## Categorical statistical summary (train)\n' + categorical_summary_md


In [ ]:
# Identify roles (Phase 1 requirement)
target_col = train_profile.inferred_target
identifier_cols = train_profile.inferred_identifier_columns
numeric_features = train_profile.inferred_numerical_features
categorical_features = train_profile.inferred_categorical_features
datetime_features = train_profile.inferred_datetime_features

roles_md = (
    '## Identified feature groups (heuristic, documented uncertainty)\n'
    + '- Target column: ' + str(target_col) + '\n'
    + '- Identifier columns: ' + (', '.join(identifier_cols) if identifier_cols else 'None detected') + '\n'
    + '- Numerical features: ' + str(len(numeric_features)) + ' (' + (', '.join(numeric_features[:20]) + ('...' if len(numeric_features) > 20 else '')) + ')' + '\n'
    + '- Categorical features: ' + str(len(categorical_features)) + ' (' + (', '.join(categorical_features[:20]) + ('...' if len(categorical_features) > 20 else '')) + ')' + '\n'
    + '- Datetime features: ' + (', '.join(datetime_features) if datetime_features else 'None detected') + '\n'
)


In [ ]:
# Cross-schema matching report (Phase 1 requirement)
cross_schema_md = (
    '## Validation dataset schema match vs training\n'
    + 'Missing columns/additional columns/dtype mismatches are reported below.\n\n'
    + issues_to_markdown(cross_schema_issues)
)


In [ ]:
# Initial Data Audit Report (Phase 1 required deliverable)
risks: list[str] = []
if any(i.issue_type == 'numeric' for i in train_issues + val_issues):
    risks.append('Invalid numeric values found; Phase 2 must strictly coerce/validate and handle failures.')
if any(i.issue_type == 'schema' for i in cross_schema_issues):
    risks.append('Train/validation schema mismatch; Phase 2 must align preprocessing and encoding consistently.')
if any((i.message or '').lower().find('duplicate') >= 0 for i in train_issues + val_issues):
    risks.append('Duplicate records/IDs detected; Phase 2 must decide whether to deduplicate and avoid leakage.')
if any((i.message or '').lower().find('missing') >= 0 for i in train_issues + val_issues):
    risks.append('Missingness present; Phase 2 must decide imputation strategy or missing-indicator usage.')
if not risks:
    risks = ['No major data-quality risks detected by heuristics; still re-check edge cases in Phase 2.']

recommendations = [
    'Validate numeric coercion policy and enforce consistent types across splits.',
    'If categorical values show whitespace/case issues, normalize categories in Phase 2 (fit on training only).',
    'If datetime columns exist, derive safe time features without leaking validation information.',
    'If coordinate-like columns exist, verify ranges and confirm correct units before modeling.',
]

data_quality_md = (
    '## Data quality issues\n'
    + '### Training issues\n'
    + issues_to_markdown(train_issues)
    + '\n\n### Validation issues\n'
    + issues_to_markdown(val_issues)
)

concise_summary_md = (
    '# Initial Data Audit Report (Phase 1)\n\n'
    + '## Concise summary\n'
    + '- Dataset dimensions (train): ' + str(train_df.shape) + '\n'
    + '- Dataset dimensions (validation): ' + str(validation_df.shape) + '\n'
    + '- Number of features (train): ' + str(train_df.shape[1]) + '\n'
    + '- Target variable (heuristic): ' + str(target_col) + '\n'
    + '- Data quality issues found: train=' + str(len(train_issues)) + ', validation=' + str(len(val_issues)) + ', cross-schema=' + str(len(cross_schema_issues)) + '\n'
    + '- Risks identified:\n'
    + ''.join(['  - ' + r + '\n' for r in risks])
    + '- Recommendations before Phase 2 (EDA):\n'
    + ''.join(['  - ' + r + '\n' for r in recommendations])
)

data_audit_md = (
    concise_summary_md
    + '\n\n## Schema & overview (train)\n'
    + 'Rows: ' + str(len(train_df)) + ', Columns: ' + str(train_df.shape[1]) + '\n'
    + 'Memory usage (deep): ' + str(train_schema.memory_bytes) + ' bytes\n'
    + '\n\n## Schema & overview (validation)\n'
    + 'Rows: ' + str(len(validation_df)) + ', Columns: ' + str(validation_df.shape[1]) + '\n'
    + 'Memory usage (deep): ' + str(val_schema.memory_bytes) + ' bytes\n'
    + '\n\n' + roles_md
    + '\n\n' + numeric_md
    + '\n\n' + categorical_md
    + '\n\n' + feature_inspection_md
    + '\n\n' + cross_schema_md
    + '\n\n' + data_quality_md
    + '\n\n## Potential preprocessing requirements (initial)
    - Missing values handling strategy depends on missingness patterns; validate before choosing imputers.
    - Invalid numeric/categorical normalization should be consistent across train and validation.
    - If categorical features are high-cardinality, plan encoding strategy carefully.
) 

# The above block is documented; the actual audit content comes from src modules.
)

print('Built data_audit_md and feature_dictionary_md')


In [ ]:
data_audit_path = REPORTS_DIR / 'data_audit.md'
feature_dictionary_path = REPORTS_DIR / 'feature_dictionary.md'

data_audit_path.write_text(data_audit_md, encoding='utf-8')
feature_dictionary_path.write_text(feature_dictionary_md, encoding='utf-8')

print('Wrote:', str(data_audit_path))
print('Wrote:', str(feature_dictionary_path))


## End of Phase 1

No EDA or modeling steps are included.